In [14]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score, log_loss

file = '/Users/alejandrogomez-paz/Desktop/UFC Project/3. models/logistic_model/features.csv'
df = pd.read_csv(file, index_col=0)
df['date'] = pd.to_datetime(df['date'])

feature_cols = [c for c in df.columns if c.endswith('_diff')]
df = df.dropna(subset=feature_cols)          # drops debut fights (~27%)

cutoff = df['date'].quantile(0.7)
train = df[df['date'] < cutoff]
test  = df[df['date'] >= cutoff]

scaler = StandardScaler().fit(train[feature_cols])   # fit on train only ✓
X_train = scaler.transform(train[feature_cols])
X_test  = scaler.transform(test[feature_cols])
y_train, y_test = train['y'], test['y']

model = LogisticRegression(solver='saga', l1_ratio=1.0, C=0.1, max_iter=5000)
model.fit(X_train, y_train)

p = model.predict_proba(X_test)[:, 1]
print(f"accuracy: {accuracy_score(y_test, p > 0.5):.3f}")
print(f"AUC:      {roc_auc_score(y_test, p):.3f}")
print(f"log loss: {log_loss(y_test, p):.3f}")

coefs = pd.Series(model.coef_[0], index=feature_cols).sort_values(key=abs, ascending=False)
print(coefs.head(10))

accuracy: 0.613
AUC:      0.659
log loss: 0.652
age_diff                        -0.349521
wins_diff                        0.209060
td_attempted_norm_diff           0.185243
rating_diff                      0.183943
head_landed_norm_diff            0.163464
losses_diff                     -0.135256
stance_Orthodox_diff            -0.101580
total_str_attempted_norm_diff   -0.091294
height_inches_z_diff            -0.086013
reach_z_diff                     0.083199
dtype: float64


In [ ]:
# --- bootstrap ensemble: confidence intervals + deployment artifact ---
# refits on ALL data (the train/test eval above validates the setup),
# then B bootstrap refits; rerun this cell whenever features.csv updates
import numpy as np
import joblib

folder = '/Users/alejandrogomez-paz/Desktop/UFC Project/3. models/logistic_model/'

def fit_pipeline(data):
    sc = StandardScaler().fit(data[feature_cols])
    m = LogisticRegression(solver='saga', l1_ratio=1.0, C=0.1, max_iter=5000)
    m.fit(sc.transform(data[feature_cols]), data['y'])
    return sc, m

scaler_full, model_full = fit_pipeline(df)   # main model, every fight

B = 300
ensemble = [fit_pipeline(df.sample(len(df), replace=True, random_state=b))
            for b in range(B)]

joblib.dump({'scaler': scaler_full, 'model': model_full,
             'ensemble': ensemble, 'feature_cols': feature_cols},
            folder + 'model.joblib')

# sanity check: bootstrap CI for one example fight
x = df[feature_cols].iloc[[-1]]
ps = np.array([m.predict_proba(sc.transform(x))[0, 1] for sc, m in ensemble])
print(f"saved {B}-model ensemble -> model.joblib")
print(f"example: p={np.median(ps):.3f}, 95% CI [{np.percentile(ps, 2.5):.3f}, {np.percentile(ps, 97.5):.3f}]")